# Check Little / Big Ending form hex log file 

In [1]:
import struct
import ast

## 1. Probedaten einlesen 

In [2]:
from pathlib import Path

# Pfad festlegen
project_path = Path.cwd().parent
folder = 'example_output'
filename = 'hex_24102025_1434'
filepath = project_path / folder / filename

# Datei einlesen
try:
    with open(filepath, 'r') as f:
        file = f.read()
except Exception as e:
    print(f"Fehler beim Lesen der Datei: {e}")

# Checken
print(f"File output: {file[:100]}")
print(type(file))

File output: b'\x00d\x00z\x00x\x00m\x00d\x00d\x00q\x00q\x00m\x00p\x00u\x00z\x00\x80\x00\x85\x00\x88\x00\x80\x00p\
<class 'str'>


In [3]:
# Umwandlung von String zu Binärdaten
try:
    # bytes-Objekt erstellen 
    row_bytes = ast.literal_eval(file)
    
    print(f"File output: {row_bytes[:50]}")
    print(type(row_bytes))
    
except (ValueError, SyntaxError) as e:
    print(f"Fehler beim Umwandeln des Strings in Binärdaten: {e}")

File output: b'\x00d\x00z\x00x\x00m\x00d\x00d\x00q\x00q\x00m\x00p\x00u\x00z\x00\x80\x00\x85\x00\x88\x00\x80\x00p\x00r\x00|\x00\x89\x00\x99\x00\xa2\x00\xa1\x00\x88\x00g'
<class 'bytes'>


## 2. Logging Funktion  

In [4]:
def format_byte(i: int, b: int) -> tuple:
    """
    Formatiert ein einzelnes Byte in verschiedene Darstellungen.
    Gibt ein Tupel (index, dec, hex, bin, ascii) in strings zurück.
    """
    index = i                          # Byte-Position im Stream
    dec = b                            # Dezimalwert (0–255)
    hx = f"{b:02X}"                    # Hexadezimalwert, immer zweistellig
    binv = f"{b:08b}"                  # Binärwert, 8 Bits
    asc = chr(b) if 32 <= b < 127 else "."  # ASCII-Zeichen, falls druckbar, sonst Punkt

    return (index, dec, hx, binv, asc)

def show_values(row_bytes: bytes) -> list:
    """
    Zeigt alle Bytes in einer Tabelle und gibt sie als Liste zurück.
    """
    values = []

    # Jedes Byte formatieren und zur Liste hinzufügen
    for i, b in enumerate(row_bytes):
        values.append(format_byte(i, b))

    # Tabelle drucken
    print(f"{'Index':<6} {'DEC':<5} {'HEX':<5} {'BIN':<10} {'ASCII'}")
    print("-" * 40)
    for index, dec, hx, binv, asc in values:
        print(f"{index:<6} {dec:<5} {hx:<5} {binv:<10} {asc}")
    print("-" * 40)

    return values

show_values(row_bytes)

Index  DEC   HEX   BIN        ASCII
----------------------------------------
0      0     00    00000000   .
1      100   64    01100100   d
2      0     00    00000000   .
3      122   7A    01111010   z
4      0     00    00000000   .
5      120   78    01111000   x
6      0     00    00000000   .
7      109   6D    01101101   m
8      0     00    00000000   .
9      100   64    01100100   d
10     0     00    00000000   .
11     100   64    01100100   d
12     0     00    00000000   .
13     113   71    01110001   q
14     0     00    00000000   .
15     113   71    01110001   q
16     0     00    00000000   .
17     109   6D    01101101   m
18     0     00    00000000   .
19     112   70    01110000   p
20     0     00    00000000   .
21     117   75    01110101   u
22     0     00    00000000   .
23     122   7A    01111010   z
24     0     00    00000000   .
25     128   80    10000000   .
26     0     00    00000000   .
27     133   85    10000101   .
28     0     00    00000000

[(0, 0, '00', '00000000', '.'),
 (1, 100, '64', '01100100', 'd'),
 (2, 0, '00', '00000000', '.'),
 (3, 122, '7A', '01111010', 'z'),
 (4, 0, '00', '00000000', '.'),
 (5, 120, '78', '01111000', 'x'),
 (6, 0, '00', '00000000', '.'),
 (7, 109, '6D', '01101101', 'm'),
 (8, 0, '00', '00000000', '.'),
 (9, 100, '64', '01100100', 'd'),
 (10, 0, '00', '00000000', '.'),
 (11, 100, '64', '01100100', 'd'),
 (12, 0, '00', '00000000', '.'),
 (13, 113, '71', '01110001', 'q'),
 (14, 0, '00', '00000000', '.'),
 (15, 113, '71', '01110001', 'q'),
 (16, 0, '00', '00000000', '.'),
 (17, 109, '6D', '01101101', 'm'),
 (18, 0, '00', '00000000', '.'),
 (19, 112, '70', '01110000', 'p'),
 (20, 0, '00', '00000000', '.'),
 (21, 117, '75', '01110101', 'u'),
 (22, 0, '00', '00000000', '.'),
 (23, 122, '7A', '01111010', 'z'),
 (24, 0, '00', '00000000', '.'),
 (25, 128, '80', '10000000', '.'),
 (26, 0, '00', '00000000', '.'),
 (27, 133, '85', '10000101', '.'),
 (28, 0, '00', '00000000', '.'),
 (29, 136, '88', '1000100

## 3 Umwandlung in Datenformate 

In [ ]:
import struct

def parse_values(row_bytes: bytes):
    """
    Wandelt Bytes in verschiedene numerische Datentypen um
    und gibt sie tabellarisch pro Index aus.
    """

    result = {}

    # --- 1. uint8 ---
    result["uint8_list"] = list(row_bytes)

    # --- 2. uint16 Little / Big ---
    if len(row_bytes) >= 2:
        n16 = len(row_bytes) // 2
        result["uint16_le"] = list(struct.unpack('<' + 'H' * n16, row_bytes[:n16 * 2]))
        result["uint16_be"] = list(struct.unpack('>' + 'H' * n16, row_bytes[:n16 * 2]))
    else:
        result["uint16_le"] = []
        result["uint16_be"] = []

    # --- 3. uint32 Little / Big ---
    if len(row_bytes) >= 4:
        n32 = len(row_bytes) // 4
        result["uint32_le"] = list(struct.unpack('<' + 'I' * n32, row_bytes[:n32 * 4]))
        result["uint32_be"] = list(struct.unpack('>' + 'I' * n32, row_bytes[:n32 * 4]))
    else:
        result["uint32_le"] = []
        result["uint32_be"] = []

    # --- 4. float32 Little / Big ---
    if len(row_bytes) >= 4:
        n32 = len(row_bytes) // 4
        result["float32_le"] = list(struct.unpack('<' + 'f' * n32, row_bytes[:n32 * 4]))
        result["float32_be"] = list(struct.unpack('>' + 'f' * n32, row_bytes[:n32 * 4]))
    else:
        result["float32_le"] = []
        result["float32_be"] = []

    #print("\n# uint8")
    print(f"\n# uint8_list: {result["uint8_list"]}")

    print(f"\n# uint16 little: {result["uint16_le"]}")
    print(f"# uint16 big: {result["uint16_be"]}")

    print(f"\n# uint32 little: {result["uint32_le"]}")
    print(f"# uint32 big: {result["uint32_be"]}")

    print(f"\n# float32 little: {result["float32_le"]}")
    print(f"# float32 big: {result["float32_be"]}")
        
    return result


In [6]:

parse_values(row_bytes)



# uint8_list: [0, 100, 0, 122, 0, 120, 0, 109, 0, 100, 0, 100, 0, 113, 0, 113, 0, 109, 0, 112, 0, 117, 0, 122, 0, 128, 0, 133, 0, 136, 0, 128, 0, 112, 0, 114, 0, 124, 0, 137, 0, 153, 0, 162, 0, 161, 0, 136, 0, 103, 0, 175, 0, 76, 1, 210, 1, 21, 2, 223, 1, 136, 1, 92, 1, 78, 1, 84, 1, 114, 1, 171, 1, 222, 1, 27, 2, 103, 2, 180, 2, 205, 2, 186, 2, 178, 2, 233, 2, 253, 2, 229, 2, 115, 2, 214, 1, 63, 1, 7, 1, 2, 1, 17, 1, 22, 1, 254, 0, 214, 0, 173, 0, 148, 0, 151, 0, 159, 0, 162, 0, 156, 0, 155, 0, 151, 0, 142, 0, 138, 0, 142, 0, 144, 0, 136, 0, 128, 0, 129, 0, 125, 0, 109, 0, 109, 0, 119, 0, 122, 0, 120, 0, 105, 0, 91, 0, 105, 0, 123, 0, 118, 0, 102, 0, 87, 0, 77, 0, 84, 0, 95, 0, 84, 0, 71, 0, 82, 0, 97, 0, 98, 0, 82, 0, 60, 0, 78, 0, 114, 0, 110, 0, 81, 0, 91, 0, 96, 0, 80, 0, 126, 0, 181, 0, 193, 0, 218, 0, 242, 0, 219, 0, 191, 0, 199, 0, 198, 0, 180, 0, 165, 0, 176, 0, 148, 0, 139, 0, 136, 0, 99, 0, 73, 0, 64, 0, 58, 0, 46, 0, 52, 0, 49, 0, 42, 0, 44, 0, 37, 0, 45, 0, 51, 0, 45, 0, 

{'uint8_list': [0,
  100,
  0,
  122,
  0,
  120,
  0,
  109,
  0,
  100,
  0,
  100,
  0,
  113,
  0,
  113,
  0,
  109,
  0,
  112,
  0,
  117,
  0,
  122,
  0,
  128,
  0,
  133,
  0,
  136,
  0,
  128,
  0,
  112,
  0,
  114,
  0,
  124,
  0,
  137,
  0,
  153,
  0,
  162,
  0,
  161,
  0,
  136,
  0,
  103,
  0,
  175,
  0,
  76,
  1,
  210,
  1,
  21,
  2,
  223,
  1,
  136,
  1,
  92,
  1,
  78,
  1,
  84,
  1,
  114,
  1,
  171,
  1,
  222,
  1,
  27,
  2,
  103,
  2,
  180,
  2,
  205,
  2,
  186,
  2,
  178,
  2,
  233,
  2,
  253,
  2,
  229,
  2,
  115,
  2,
  214,
  1,
  63,
  1,
  7,
  1,
  2,
  1,
  17,
  1,
  22,
  1,
  254,
  0,
  214,
  0,
  173,
  0,
  148,
  0,
  151,
  0,
  159,
  0,
  162,
  0,
  156,
  0,
  155,
  0,
  151,
  0,
  142,
  0,
  138,
  0,
  142,
  0,
  144,
  0,
  136,
  0,
  128,
  0,
  129,
  0,
  125,
  0,
  109,
  0,
  109,
  0,
  119,
  0,
  122,
  0,
  120,
  0,
  105,
  0,
  91,
  0,
  105,
  0,
  123,
  0,
  118,
  0,
  102,
  0,
  87,
  0,
